In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e3/train.csv
/kaggle/input/competitions/playground-series-s6e3/test.csv


In [ ]:
!pip -q install -U "numpy==1.26.4" "scipy==1.11.4" "scikit-learn==1.4.2" "pandas==2.2.2"

In [ ]:
!pip -q install -U xgboost lightgbm catboost

In [ ]:
!pip -q install --no-deps "numpy==1.26.4"
!pip -q install -q "scipy==1.11.4" "scikit-learn==1.4.2" "pandas==2.2.2"

In [ ]:
!pip -q install xgboost lightgbm catboost
!pip -q install git+https://github.com/PerforatedAI/PerforatedAI.git

In [2]:
# =========================================================
# TELCO CHURN
# CONTRACT-AWARE CV + SIGNAL FEATURES + MULTI-SEED XGB
# + PerforatedAI DENDRITES (Embedding-MLP)
# + OOF STACKING BLEND (XGB + PAI)
# =========================================================

import numpy as np
import pandas as pd
import gc
import warnings, time
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from scipy.special import logit, expit
from scipy.stats import rankdata
from xgboost import XGBClassifier

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# PerforatedAI
!pip -q install git+https://github.com/PerforatedAI/PerforatedAI.git
from perforatedai import globals_perforatedai as GPA
from perforatedai import utils_perforatedai as UPA

# ======================
# LOAD DATA
# ======================
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/train.csv")
test  = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/test.csv")

TARGET = "Churn"
ID_COL = "id"

train[TARGET] = train[TARGET].map({"No":0,"Yes":1}).astype(int)

# total charges numeric fix
train["TotalCharges"] = pd.to_numeric(train["TotalCharges"], errors="coerce")
test["TotalCharges"]  = pd.to_numeric(test["TotalCharges"], errors="coerce")
train["TotalCharges"].fillna(train["TotalCharges"].median(), inplace=True)
test["TotalCharges"].fillna(test["TotalCharges"].median(), inplace=True)

y = train[TARGET].astype(int).values
test_ids = test[ID_COL].values

X = train.drop(columns=[TARGET, ID_COL]).copy()
X_test = test.drop(columns=[ID_COL]).copy()

# ======================
# SIGNAL FEATURE ENGINEERING (UNCHANGED)
# ======================

# Core behaviour
X["ChargePerTenure"] = X["TotalCharges"] / (X["tenure"] + 1)
X_test["ChargePerTenure"] = X_test["TotalCharges"] / (X_test["tenure"] + 1)

# Log transforms
for col in ["MonthlyCharges", "TotalCharges"]:
    X[f"log_{col}"] = np.log1p(X[col])
    X_test[f"log_{col}"] = np.log1p(X_test[col])

# Service count
service_cols = [
    "OnlineSecurity","OnlineBackup","DeviceProtection",
    "TechSupport","StreamingTV","StreamingMovies"
]
for col in service_cols:
    if col in X.columns:
        X[col] = (X[col] == "Yes").astype(int)
        X_test[col] = (X_test[col] == "Yes").astype(int)

X["ServiceCount"] = X[service_cols].sum(axis=1)
X_test["ServiceCount"] = X_test[service_cols].sum(axis=1)

# Contract interaction
X["Monthly_x_Contract"] = X["MonthlyCharges"] * (X["Contract"] == "Month-to-month").astype(int)
X_test["Monthly_x_Contract"] = X_test["MonthlyCharges"] * (X_test["Contract"] == "Month-to-month").astype(int)

# ======================
# CONTRACT-AWARE STRATIFICATION
# ======================
stratify_key = train["Churn"].astype(str) + "_" + train["Contract"].astype(str)
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# =========================================================
# PART A) XGB FEATURES (ONE-HOT)
# =========================================================
X_enc = pd.get_dummies(X, drop_first=True)
X_test_enc = pd.get_dummies(X_test, drop_first=True)
X_enc, X_test_enc = X_enc.align(X_test_enc, join="left", axis=1, fill_value=0)

X_enc = X_enc.astype("float32")
X_test_enc = X_test_enc.astype("float32")

gc.collect()

# =========================================================
# PART B) PAI FEATURES (EMBEDDINGS FOR CATEGORICALS + SCALED NUMERICS)
# =========================================================
# Identify categorical columns in the *original engineered frame*
cat_cols = [c for c in X.columns if X[c].dtype == "object"]
num_cols = [c for c in X.columns if c not in cat_cols]

# Fill missing for cats + nums
for c in cat_cols:
    X[c] = X[c].astype("object").fillna("NA")
    X_test[c] = X_test[c].astype("object").fillna("NA")
for c in num_cols:
    X[c] = pd.to_numeric(X[c], errors="coerce").fillna(X[c].median()).astype(np.float32)
    X_test[c] = pd.to_numeric(X_test[c], errors="coerce").fillna(X[c].median()).astype(np.float32)

# Label-encode categoricals jointly
X_cat = np.zeros((len(X), len(cat_cols)), dtype=np.int64)
T_cat = np.zeros((len(X_test), len(cat_cols)), dtype=np.int64)
cat_sizes = []
for j, c in enumerate(cat_cols):
    all_vals = pd.concat([X[c], X_test[c]], axis=0, ignore_index=True)
    codes, uniques = pd.factorize(all_vals, sort=True)
    X_cat[:, j] = codes[:len(X)]
    T_cat[:, j] = codes[len(X):]
    cat_sizes.append(int(len(uniques)))

# Scale numerics
scaler = StandardScaler()
X_num = scaler.fit_transform(X[num_cols].values.astype(np.float32)).astype(np.float32)
T_num = scaler.transform(X_test[num_cols].values.astype(np.float32)).astype(np.float32)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("XGB dims:", X_enc.shape, "PAI dims: num=", X_num.shape[1], "cat=", X_cat.shape[1])

# =========================================================
# PerforatedAI non-interactive config
# =========================================================
if hasattr(GPA, "pc"):
    if hasattr(GPA.pc, "set_unwrapped_modules_confirmed"):
        GPA.pc.set_unwrapped_modules_confirmed(True)
    if hasattr(GPA.pc, "set_testing_dendrite_capacity"):
        GPA.pc.set_testing_dendrite_capacity(False)
    if hasattr(GPA.pc, "set_weight_decay_accepted"):
        GPA.pc.set_weight_decay_accepted(True)

    # Track fragile modules, DO NOT track Linear (we want dendrites on linear layers)
    try:
        trk = GPA.pc.get_module_names_to_track()
        trk.add("Embedding")
        trk.add("LayerNorm")
        trk.add("BatchNorm1d")
    except Exception:
        pass

    # Switching strategy: moderate (fast + stable)
    if hasattr(GPA.pc, "set_switch_mode") and hasattr(GPA.pc, "DOING_FIXED_SWITCH"):
        GPA.pc.set_switch_mode(GPA.pc.DOING_FIXED_SWITCH)
    if hasattr(GPA.pc, "set_fixed_switch_num"):
        GPA.pc.set_fixed_switch_num(2)

def emb_dim(card):
    return int(min(16, max(2, round(card ** 0.5))))

class PAIEmbMLP(nn.Module):
    def __init__(self, n_num, cat_sizes, dropout=0.15):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(s, emb_dim(s)) for s in cat_sizes])
        cat_out = sum(e.embedding_dim for e in self.embs)
        d_in = n_num + cat_out
        self.bn0 = nn.BatchNorm1d(d_in)
        self.fc1 = nn.Linear(d_in, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.fc2 = nn.Linear(256, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.fc3 = nn.Linear(128, 64)
        self.out = nn.Linear(64, 1)
        self.drop = nn.Dropout(dropout)

    def forward(self, x_num, x_cat):
        if len(self.embs) > 0:
            em = [emb(x_cat[:, i]) for i, emb in enumerate(self.embs)]
            x = torch.cat([x_num] + em, dim=1)
        else:
            x = x_num
        x = self.bn0(x)
        x = self.drop(F.relu(self.bn1(self.fc1(x))))
        x = self.drop(F.relu(self.bn2(self.fc2(x))))
        x = self.drop(F.relu(self.fc3(x)))
        return self.out(x).squeeze(-1)

def set_opt(opt):
    if hasattr(GPA.pai_tracker, "set_optimizer_instance"):
        GPA.pai_tracker.set_optimizer_instance(opt)
    else:
        GPA.pai_tracker.setOptimizerInstance(opt)

def train_pai_fold(tr_idx, va_idx, epochs=4, bs=16384, lr=1e-3):
    ds_tr = TensorDataset(
        torch.from_numpy(X_num[tr_idx]),
        torch.from_numpy(X_cat[tr_idx]),
        torch.from_numpy(y[tr_idx].astype(np.float32)),
    )
    ds_va = TensorDataset(
        torch.from_numpy(X_num[va_idx]),
        torch.from_numpy(X_cat[va_idx]),
        torch.from_numpy(y[va_idx].astype(np.float32)),
    )
    dl_tr = DataLoader(ds_tr, batch_size=bs, shuffle=True, num_workers=0)
    dl_va = DataLoader(ds_va, batch_size=bs, shuffle=False, num_workers=0)

    base = PAIEmbMLP(n_num=X_num.shape[1], cat_sizes=cat_sizes).to(device)
    model = UPA.initialize_pai(base).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=0.0)
    set_opt(opt)
    scaler_amp = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

    best_auc = -1.0
    best_va = None

    for ep in range(1, epochs + 1):
        model.train()
        for xn, xc, yy in dl_tr:
            xn, xc, yy = xn.to(device), xc.to(device), yy.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
                logits = model(xn, xc)
                loss = F.binary_cross_entropy_with_logits(logits, yy)
            scaler_amp.scale(loss).backward()
            scaler_amp.step(opt)
            scaler_amp.update()

        model.eval()
        vp, vt = [], []
        with torch.no_grad():
            for xn, xc, yy in dl_va:
                xn, xc = xn.to(device), xc.to(device)
                vp.append(torch.sigmoid(model(xn, xc)).cpu().numpy())
                vt.append(yy.numpy())
        vp = np.concatenate(vp)
        vt = np.concatenate(vt)
        auc = roc_auc_score(vt, vp)
        if auc > best_auc:
            best_auc = auc
            best_va = vp.copy()

        # Let PAI restructure if it wants
        model, restructured, done = GPA.pai_tracker.add_validation_score(float(auc), model)
        model = model.to(device)
        if restructured:
            opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=0.0)
            set_opt(opt)
        if done:
            break

    return best_va, model, best_auc

@torch.no_grad()
def predict_pai(model, bs=32768):
    ds = TensorDataset(torch.from_numpy(T_num), torch.from_numpy(T_cat))
    dl = DataLoader(ds, batch_size=bs, shuffle=False, num_workers=0)
    model.eval()
    out = []
    for xn, xc in dl:
        xn, xc = xn.to(device), xc.to(device)
        out.append(torch.sigmoid(model(xn, xc)).cpu().numpy())
    return np.concatenate(out).astype(np.float32)

# =========================================================
# TRAINING: XGB (multi-seed) + PAI (single pass CV)
# =========================================================
seeds = [42, 77, 99]

# XGB OOF + test logits (logit-average)
oof_logits_xgb = np.zeros(len(X_enc), dtype=np.float64)
test_logits_xgb = np.zeros(len(X_test_enc), dtype=np.float64)

# PAI OOF + test (probabilities)
oof_pai = np.zeros(len(X_enc), dtype=np.float32)
test_pai = np.zeros(len(X_test_enc), dtype=np.float32)

# ----- XGB multi-seed -----
for seed in seeds:
    print(f"\n===== XGB Seed {seed} =====")
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_enc, stratify_key), 1):
        X_train, X_val = X_enc.iloc[train_idx], X_enc.iloc[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        model = XGBClassifier(
            n_estimators=50000,
            learning_rate=0.01,
            max_depth=4,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=8,
            reg_alpha=2,
            min_child_weight=4,
            gamma=0.2,
            objective="binary:logistic",
            eval_metric="auc",
            tree_method="hist",
            device="cuda",
            early_stopping_rounds=1200,
            random_state=seed,          # ✅ THIS MAKES SEEDS ACTUALLY DIFFERENT
        )

        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

        val_pred = model.predict_proba(X_val)[:, 1]
        test_pred = model.predict_proba(X_test_enc)[:, 1]

        oof_logits_xgb[val_idx] += logit(np.clip(val_pred, 1e-6, 1-1e-6)) / len(seeds)
        test_logits_xgb += logit(np.clip(test_pred, 1e-6, 1-1e-6)) / (len(seeds) * N_SPLITS)

        print(f"XGB Seed {seed} Fold {fold} AUC:", roc_auc_score(y_val, val_pred))

# Convert XGB logits -> probs
oof_xgb = expit(oof_logits_xgb).astype(np.float32)
test_xgb = expit(test_logits_xgb).astype(np.float32)
print("\nXGB FINAL CV AUC:", roc_auc_score(y, oof_xgb))

# ----- PAI CV (same folds) -----
print("\n===== PAI DENDRITES CV =====")
for fold, (train_idx, val_idx) in enumerate(skf.split(X_enc, stratify_key), 1):
    va_pred, pai_model, auc_p = train_pai_fold(train_idx, val_idx, epochs=4, bs=16384, lr=1e-3)
    oof_pai[val_idx] = va_pred.astype(np.float32)
    test_pai += predict_pai(pai_model, bs=32768) / N_SPLITS
    print(f"PAI Fold {fold} AUC:", roc_auc_score(y[val_idx], va_pred))

print("\nPAI FINAL CV AUC:", roc_auc_score(y, oof_pai))

# =========================================================
# BLEND: OOF stacking (XGB + PAI)
# =========================================================
X_meta = np.column_stack([oof_xgb, oof_pai])
T_meta = np.column_stack([test_xgb, test_pai])

meta = LogisticRegression(max_iter=2000)
meta.fit(X_meta, y)
oof_blend = meta.predict_proba(X_meta)[:, 1].astype(np.float32)
test_blend = meta.predict_proba(T_meta)[:, 1].astype(np.float32)

print("\nMETA weights:", meta.coef_)
print("BLEND FINAL CV AUC:", roc_auc_score(y, oof_blend))

# Optional: mild sharpening (safe for AUC)
temperature = 1.02
test_blend = expit(logit(np.clip(test_blend, 1e-6, 1-1e-6)) * temperature).astype(np.float32)

# =========================================================
# SUBMISSION
# =========================================================
# AUC is rank-invariant, but ranking can be stable on some leaderboards.
USE_RANK = True

if USE_RANK:
    churn_pred = rankdata(test_blend) / len(test_blend)
else:
    churn_pred = test_blend

submission = pd.DataFrame({"id": test_ids, "Churn": churn_pred})
submission.to_csv("submission.csv", index=False)
print("Submission created successfully 🚀  -> submission.csv")

  Preparing metadata (setup.py) ... done
Building dendrites without Perforated Backpropagation
device: cpu
XGB dims: (594194, 29) PAI dims: num= 15 cat= 9

===== XGB Seed 42 =====
XGB Seed 42 Fold 1 AUC: 0.9177480326564388
XGB Seed 42 Fold 2 AUC: 0.9163772669339012
XGB Seed 42 Fold 3 AUC: 0.9164553000202533
XGB Seed 42 Fold 4 AUC: 0.9179936343262107
XGB Seed 42 Fold 5 AUC: 0.9155588600043327

===== XGB Seed 77 =====
XGB Seed 77 Fold 1 AUC: 0.9177312351171663
XGB Seed 77 Fold 2 AUC: 0.9163625278469747
XGB Seed 77 Fold 3 AUC: 0.9164535644328016
XGB Seed 77 Fold 4 AUC: 0.9179814906922883
XGB Seed 77 Fold 5 AUC: 0.9155424431359551

===== XGB Seed 99 =====
XGB Seed 99 Fold 1 AUC: 0.9177291257361323
XGB Seed 99 Fold 2 AUC: 0.9163331777050023
XGB Seed 99 Fold 3 AUC: 0.9164448029016652
XGB Seed 99 Fold 4 AUC: 0.9180081638349106
XGB Seed 99 Fold 5 AUC: 0.9155677034307915

XGB FINAL CV AUC: 0.9168270840707551

===== PAI DENDRITES CV =====
Running Dendrite Experiment
Adding validation score 0.906